# Iraq Farms Health — publish the crop layer (read-only on your run)
Turns every **finished** WorldCereal tile in Drive into the files the website serves from `crops/2026/`:

* `cells/*.tif`: the crop layer at ~10 m, in ~78 km blocks (cloud-optimised GeoTIFFs; QGIS, GeoLibre, the website and Iraq Farms Health all stream these)
* `overview.tif`: whole-country view at 76 m
* `iraq_crops.vrt`: one national mosaic link for QGIS / GDAL
* `stats.json` + `stats_*.csv`: wheat/barley hectares by **district → governorate → region**
* `boundaries.geojson`, `iraq_crops.qml` (QGIS colours), `legend.csv`, `index.json`

**Safe to run any time, in a second tab, while batches are still running.** It only reads the run folder. Run it again whenever more tiles finish; the website shows the numbers as *provisional* until every planned tile is done.

**Merged runs (Nov 2026):** the map is built from up to three runs. For each tile the first run that has it finished wins: the **north run** (`iraq_winter2026_north`, later season), then the **season pilot** (its 3 northern tiles only), then the **main run** (`iraq_winter2026`, calendar season). A run that doesn't exist yet is skipped. `sources.json` records which run each tile came from; the 2021 comparison reads it.

**Season pilot on its own:** set `PILOT = True` in the first cell to publish the pilot run (`iraq_pilot_season_jun`) to `crops/2026p/` instead. It never touches `crops/2026/` or the main run.

Takes about 1–2 minutes per 10 finished tiles. Run the cells in order; Google will ask for Drive access — tick every box.

In [ ]:
PILOT        = False               # True = publish the season pilot instead of the main run
RUN_NAME     = "iraq_winter2026"   # same as in the batch notebook
SEASON_LABEL = "2025-26"           # shown on the website
DATA_FOLDER  = "2026"              # becomes crops/2026/ in the repo
# Runs laid over the main run, first wins (Nov 2026). (folder, tiles or None for all its finished tiles)
OVERLAYS     = [("iraq_winter2026_north", None),                       # later season, tiles north of 34.8°N
                ("iraq_pilot_season_jun", ["irq_38N_E0240_N4040_40k",   # pilot: only its 3 northern tiles
                                           "irq_38N_E0280_N4040_40k",   #   (the Najaf control tile stays on
                                           "irq_38N_E0360_N4000_40k"])] #    the main run's calendar season)
if PILOT:                          # season pilot (Nov 2026): its own run folder and its own repo folder
    RUN_NAME     = "iraq_pilot_season_jun"
    SEASON_LABEL = "2025-26 (season pilot: Nov–Jun)"
    DATA_FOLDER  = "2026p"         # becomes crops/2026p/ in the repo, beside crops/2026/
    OVERLAYS     = []              # the pilot on its own

from google.colab import drive
drive.mount("/content/drive")
import os, shutil
RUN = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{RUN_NAME}"
OUT = f"/content/publish/{DATA_FOLDER}"
DRIVE_OUT = f"{RUN}/publish/{DATA_FOLDER}"
print("Run folder found:", os.path.isdir(RUN), RUN)
print(f"Publishing to crops/{DATA_FOLDER}/" + ("  🧪 SEASON PILOT" if PILOT else ""))
OVERLAY_ARGS = []
for name, tiles in OVERLAYS:
    p = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{name}"
    if os.path.isdir(p):
        OVERLAY_ARGS += ["--overlay", p + ("::" + ",".join(tiles) if tiles else "")]
        print(f"  overlay: {name}" + (f" ({len(tiles)} tiles)" if tiles else " (all finished tiles)"))
    else:
        print(f"  overlay: {name} — not in Drive yet, skipped")
import shlex
OVERLAY_ARGS = " ".join(shlex.quote(a) for a in OVERLAY_ARGS)
!pip -q install rasterio

In [ ]:
%%writefile /content/publish_crops.py
"""Publish finished WorldCereal tiles as an open, streamable crop layer + statistics.

python publish_crops.py <run_folder> <out_folder> [--overlay RUN[::tile,tile]]... [--season-label 2025-26] [--adm1 f --adm2 f]

Reads (never changes) the run folders: plan/grid_all.geojson, batch_*/job_tracking.csv,
batch_*/<tile>/*.tif (raw WorldCereal output) or compact/<tile>.tif.
<run_folder> is the main run: its plan lists every tile. Each --overlay is another run (e.g. the north
re-run with a later season); its finished tiles replace the main run's. The first overlay given wins.
"::tile,tile" limits an overlay to those tiles. sources.json records the run behind each tile.

Writes <out_folder>/  (copy its contents to the repo folder crops/<year>/):
  cells/z9_<x>_<y>.tif   ~10 m crop codes, EPSG:3857 COGs, one per web-map tile z9 (≈78 km)
  overview.tif           national COG at ~76 m (z11) for whole-country viewing
  iraq_crops.vrt         one national mosaic of all cells (QGIS / GDAL)
  index.json             cells, legend, provenance (read by the web page)
  stats.json, stats_*.csv  hectares by district / governorate / region
  boundaries.geojson     simplified governorates used for the statistics

Pixel codes (one band, uint8, 0 = not cropland / no data):
  1 + WorldCereal class index            WorldCereal's own classification (its default output)
  +100 "less sure": the winning crop's probability is below CONF_THRESHOLD (60 %), e.g. wheat vs barley unclear
  +200 "doubtful cereal": a cereal class (wheat/barley/rye/oats) whose wheat + barley probability
       P(wheat)+P(barley) is below CEREAL_THRESHOLD (70 %), or that sits in a speck of confident
       cereal smaller than MIN_PATCH_HA (0.1 ha, ~10 pixels, 8-connected). Confident cereal = cereal class and code < 200.
"""
import argparse
import csv
import datetime
import glob
import json
import math
import os
import re
import unicodedata
from collections import defaultdict

import numpy as np
from scipy import ndimage
import rasterio
import rasterio.shutil
from xml.sax.saxutils import quoteattr
from rasterio.features import rasterize
from rasterio.io import MemoryFile
from rasterio.transform import Affine
from rasterio.warp import Resampling, reproject, transform_bounds, transform_geom

CONF_THRESHOLD = 60
CEREAL_THRESHOLD = 70
MIN_PATCH_HA = 0.1     # confident-cereal clusters smaller than this are treated as doubtful (speckle, not fields)
CEREALS = ("wheat", "barley", "rye", "oats")
# the 70 % test uses wheat + barley only: every tile has these (compact copies of batch 1 have no rye/oats),
# so the rule is the same for every tile whether or not its raw file still exists
PROB_CLASSES = ("wheat", "barley")
PALETTE = {"wheat": "#E3B505", "barley": "#C2571A", "rye": "#B98B3E", "oats": "#D8C27A", "maize": "#F08A24",
           "sorghum": "#A0522D", "rice": "#5DADE2", "other_temporary_crops": "#8FBC8F", "millet": "#CDAA7D",
           "vegetables": "#4F9D69", "beet": "#A64D79", "dry_pulses_legumes": "#8E6CB0", "sunflower": "#FFD23F",
           "soy_soybeans": "#9ACD32", "rapeseed_rape": "#D4D400", "other_oilseed": "#BDB76B", "groundnuts": "#C68E5E",
           "fibre_crops": "#B8B8D8", "potatoes": "#9C6644", "cassava": "#6B8E23", "tobacco": "#556B2F",
           "grass_fodder_crops": "#7FC97F", "sugar_cane": "#3CB371", "no_crop": "#9AA0A6"}
CELL_Z, MAX_Z, OVERVIEW_Z = 9, 14, 11
CELL_PX = 6656                      # 26 x 256 px per cell → 11.76 m in Web Mercator ≈ 9.8 m on the ground in Iraq (WorldCereal is 10 m)
R = 20037508.342789244
LESS = 100
COG_OPTS = dict(driver="COG", BLOCKSIZE=256, COMPRESS="DEFLATE", LEVEL=9, OVERVIEW_RESAMPLING="MODE", BIGTIFF="NO")

ap = argparse.ArgumentParser()
ap.add_argument("run"); ap.add_argument("out")
ap.add_argument("--season-label", default="2025-26")
ap.add_argument("--adm1"); ap.add_argument("--adm2")
ap.add_argument("--adm-name-field", default="shapeName")
ap.add_argument("--skip-cells", action="store_true", help="statistics only")
ap.add_argument("--overlay", action="append", default=[],
                help="another run whose finished tiles replace the main run's (RUN or RUN::tile,tile); first wins")
A = ap.parse_args()
os.makedirs(os.path.join(A.out, "cells"), exist_ok=True)


# ----------------------------------------------------------------------------- inputs
def finished_tiles(run):
    """tile -> (kind, path) for every finished tile with a file on disk."""
    done = set()
    for b in sorted(glob.glob(os.path.join(run, "batch_[0-9][0-9]", "job_tracking.csv"))):
        for r in csv.DictReader(open(b, newline="")):
            if r.get("status") == "finished":
                done.add(r["tile_name"])
    out = {}
    for f in glob.glob(os.path.join(run, "compact", "*.tif")):
        t = os.path.basename(f)[:-4]
        if t in done:
            out[t] = ("compact", f)
    for f in glob.glob(os.path.join(run, "batch_[0-9][0-9]", "*", "*.tif")):
        t = os.path.basename(os.path.dirname(f))
        if t in done:
            out.setdefault(t, ("raw", f))
    return out


def read_tile(kind, f):
    """-> codes (uint8), classes, crs, transform."""
    with rasterio.open(f) as ds:
        names = list(ds.descriptions)
        if kind == "compact":
            cls, conf = ds.read(1), ds.read(2)
            classes = json.loads(ds.tags()["classes"])
            psum = ds.read(3).astype(np.int16) + ds.read(4).astype(np.int16)    # bands 3-4 = p_wheat, p_barley
        else:
            season = next(n.split(":")[1] for n in names if n.startswith("croptype_classification:"))
            classes = [n.split(":")[2] for n in names if n.startswith(f"croptype_probability:{season}:")]
            cls = ds.read(names.index(f"croptype_classification:{season}") + 1)
            conf = ds.read(names.index(f"croptype_probability:{season}") + 1)
            psum = sum(ds.read(names.index(f"croptype_probability:{season}:{c}") + 1).astype(np.int16)
                       for c in PROB_CLASSES if c in classes)
        valid = cls < len(classes)
        cereal = np.isin(cls, [i for i, c in enumerate(classes) if c in CEREALS])
        codes = np.zeros(cls.shape, np.uint8)
        codes[valid] = cls[valid].astype(np.uint8) + 1
        codes[valid & (conf < CONF_THRESHOLD)] += LESS
        doubtful = valid & cereal & (psum < CEREAL_THRESHOLD)
        doubtful |= small_patches(valid & cereal & ~doubtful, ds.transform)
        codes[doubtful] = cls[doubtful].astype(np.uint8) + 1 + 2 * LESS
        return codes, classes, ds.crs, ds.transform


def small_patches(mask, transform, min_ha=MIN_PATCH_HA):
    """Pixels of `mask` that belong to 8-connected clusters smaller than min_ha."""
    min_px = math.ceil(min_ha * 1e4 / abs(transform.a * transform.e))
    lab, n = ndimage.label(mask, structure=np.ones((3, 3), bool))
    if not n:
        return np.zeros(mask.shape, bool)
    size = np.bincount(lab.ravel())
    small = size < min_px
    small[0] = False
    return small[lab]


plan = json.load(open(os.path.join(A.run, "plan", "grid_all.geojson")))
plan_tiles = {f["properties"]["tile_name"]: f for f in plan["features"]}


def run_seasons(run):
    """tile -> [season start, season end] from a run's own plan."""
    p = os.path.join(run, "plan", "grid_all.geojson")
    if not os.path.exists(p):
        return {}
    return {f["properties"]["tile_name"]: [str(f["properties"].get("season_start_tc-s1"))[:10],
                                           str(f["properties"].get("season_end_tc-s1"))[:10]]
            for f in json.load(open(p))["features"]}


# overlays first (first given wins), then the main run
runs = []
for o in A.overlay:
    path, _, only = o.partition("::")
    runs.append((path.rstrip("/"), set(only.split(",")) if only else None))
runs.append((A.run.rstrip("/"), None))
tiles, sources, by_run = {}, {}, defaultdict(int)
for path, only in runs:
    seasons = run_seasons(path)
    found = finished_tiles(path)
    if only is not None:
        missing = sorted(only - set(found))
        if missing:
            print(f"  {os.path.basename(path)}: not finished, so not used: {', '.join(missing)}")
        found = {t: v for t, v in found.items() if t in only}
    stray = sorted(set(found) - set(plan_tiles))
    if stray:
        raise SystemExit(f"{os.path.basename(path)} has tiles that are not in the main plan: {stray[:5]}")
    for t, v in found.items():
        if t in tiles:
            continue
        tiles[t] = v
        sources[t] = {"run": os.path.basename(path), "kind": v[0], "path": v[1], "season": seasons.get(t)}
        by_run[os.path.basename(path)] += 1
print(f"Finished tiles with files: {len(tiles)} of {len(plan_tiles)} planned")
for r, n in by_run.items():
    print(f"  from {r}: {n}")
if not tiles:
    raise SystemExit("Nothing to publish yet.")
json.dump(sources, open(os.path.join(A.out, "sources.json"), "w"), indent=1)


# ----------------------------------------------------------------------------- web-mercator helpers
def tile_size(z):
    return 2 * R / 2 ** z


def tiles_for_bounds(b, z):
    s = tile_size(z)
    x0, x1 = int((b[0] + R) // s), int((b[2] + R) // s)
    y0, y1 = int((R - b[3]) // s), int((R - b[1]) // s)
    return [(x, y) for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)]


CELL_RES = tile_size(CELL_Z) / CELL_PX


def cell_transform(x, y):
    return Affine(CELL_RES, 0, -R + x * tile_size(CELL_Z), 0, -CELL_RES, R - y * tile_size(CELL_Z))


def _tint(hexc, f):
    r, g, b = (int(hexc[i:i + 2], 16) for i in (1, 3, 5))
    return tuple(int(v + (255 - v) * f) for v in (r, g, b)) + (255,)


def cog_colormap():
    """The colours of iraq_crops.qml, so COG viewers without a style file (GeoLibre etc.) show the classes."""
    cm = {v: (0, 0, 0, 0) for v in range(256)}
    for i, c in enumerate(classes_ref or [], 1):
        col = PALETTE.get(c, "#888888")
        cm[i] = _tint(col, 0)
        cm[i + LESS] = _tint(col, 0.55)
        if c in CEREALS:
            cm[i + 2 * LESS] = _tint(col, 0.8)
    return cm


def write_cog(arr, transform, path):
    prof = dict(driver="GTiff", width=arr.shape[1], height=arr.shape[0], count=1, dtype="uint8",
                crs="EPSG:3857", transform=transform, nodata=None)
    # The zoomed-out levels are built first, with no nodata: 0 ("not cropland") must count as a class there,
    # otherwise MODE ignores it and a mostly-desert block shrinks into "crop" (the map grows as you zoom out).
    # Only then is 0 declared nodata (transparent) and the colour table added (Nov 2026), so COG viewers show
    # the classes on a clear background; the COG keeps the levels built above (OVERVIEWS=FORCE_USE_EXISTING).
    # Levels: halve until the last one is under one 256 px block, as the COG driver itself does.
    factors, k = [], 2
    while min(arr.shape) / (k // 2) > 256:
        factors.append(k); k *= 2
    with MemoryFile() as mem:
        with mem.open(**prof) as tmp:
            tmp.write(arr, 1)
            tmp.update_tags(**TAGS)
            tmp.build_overviews(factors, Resampling.mode)
            tmp.nodata = 0
            if classes_ref:
                tmp.write_colormap(1, cog_colormap())
            rasterio.shutil.copy(tmp, path, **COG_OPTS, OVERVIEWS="FORCE_USE_EXISTING")


# ----------------------------------------------------------------------------- build cells
tile_bounds_3857 = {}
classes_ref = None
for t, (kind, f) in tiles.items():
    with rasterio.open(f) as ds:
        tile_bounds_3857[t] = transform_bounds(ds.crs, "EPSG:3857", *ds.bounds)
cells = defaultdict(list)
for t, b in tile_bounds_3857.items():
    for c in tiles_for_bounds(b, CELL_Z):
        cells[c].append(t)

TAGS = {}
cell_index = []
ov_tiles = tiles_for_bounds((min(b[0] for b in tile_bounds_3857.values()), min(b[1] for b in tile_bounds_3857.values()),
                             max(b[2] for b in tile_bounds_3857.values()), max(b[3] for b in tile_bounds_3857.values())), OVERVIEW_Z)
ox0, oy0 = min(x for x, _ in ov_tiles), min(y for _, y in ov_tiles)
ox1, oy1 = max(x for x, _ in ov_tiles), max(y for _, y in ov_tiles)
ov = np.zeros(((oy1 - oy0 + 1) * 256, (ox1 - ox0 + 1) * 256), np.uint8)
ov_tr = Affine(tile_size(OVERVIEW_Z) / 256, 0, -R + ox0 * tile_size(OVERVIEW_Z), 0, -tile_size(OVERVIEW_Z) / 256, R - oy0 * tile_size(OVERVIEW_Z))

if not A.skip_cells:
    cache = {}
    n = CELL_PX
    for i, ((x, y), ts) in enumerate(sorted(cells.items())):
        arr = np.zeros((n, n), np.uint8)
        tr = cell_transform(x, y)
        for t in ts:
            if t not in cache:
                cache.clear()                      # keep at most one tile in memory
                cache[t] = read_tile(*tiles[t])
            codes, classes, crs, ttr = cache[t]
            classes_ref = classes_ref or classes
            part = np.zeros_like(arr)
            reproject(codes, part, src_transform=ttr, src_crs=crs, dst_transform=tr, dst_crs="EPSG:3857",
                      src_nodata=0, dst_nodata=0, resampling=Resampling.nearest)
            m = (arr == 0) & (part != 0)
            arr[m] = part[m]
        if not arr.any():
            continue
        if not TAGS:
            TAGS = {"classes": json.dumps(classes_ref), "less_sure_offset": str(LESS),
                    "confidence_threshold": str(CONF_THRESHOLD), "nodata_meaning": "0 = not cropland or not processed",
                    "source": "ESA WorldCereal default model (worldcereal-classification), run by Iraq Farms Health",
                    "license": "CC BY 4.0", "season": A.season_label}
        path = os.path.join(A.out, "cells", f"z{CELL_Z}_{x}_{y}.tif")
        write_cog(arr, tr, path)
        # national overview: most common value, "not cropland" included
        reproject(arr, ov, src_transform=tr, src_crs="EPSG:3857", dst_transform=ov_tr, dst_crs="EPSG:3857",
                  src_nodata=None, dst_nodata=None, resampling=Resampling.mode, init_dest_nodata=False)
        mb = os.path.getsize(path) / 1e6
        cell_index.append({"z": CELL_Z, "x": x, "y": y, "file": f"cells/z{CELL_Z}_{x}_{y}.tif", "mb": round(mb, 2),
                           "bounds_3857": [tr.c, tr.f - n * tr.a, tr.c + n * tr.a, tr.f]})
        print(f"  cell {i + 1}/{len(cells)} z{CELL_Z}/{x}/{y}: {len(ts)} tile(s) → {mb:.1f} MB" + ("  ⚠️ over 95 MB" if mb > 95 else ""))
    write_cog(ov, ov_tr, os.path.join(A.out, "overview.tif"))
    print(f"  overview.tif: {os.path.getsize(os.path.join(A.out, 'overview.tif')) / 1e6:.1f} MB")

    # national VRT (relative paths) + one VRT per overview level, so QGIS/GDAL stay fast when zoomed out
    n = CELL_PX
    res = CELL_RES
    xs = [c["x"] for c in cell_index]; ys = [c["y"] for c in cell_index]
    vx0, vy0 = min(xs), min(ys)
    t0 = cell_transform(vx0, vy0)
    with rasterio.open(os.path.join(A.out, cell_index[0]["file"])) as ds:
        wkt = ds.crs.to_wkt()
        n_ovr = len(ds.overviews(1))

    def vrt_xml(level, overviews=""):
        m = n >> level
        W, H = (max(xs) - vx0 + 1) * m, (max(ys) - vy0 + 1) * m
        r = res * (n / m)
        oo = f'<OpenOptions><OOI key="OVERVIEW_LEVEL">{level - 1}</OOI></OpenOptions>' if level else ""
        srcs = "".join(
            f'<SimpleSource><SourceFilename relativeToVRT="1">{c["file"]}</SourceFilename>{oo}<SourceBand>1</SourceBand>'
            f'<SourceProperties RasterXSize="{m}" RasterYSize="{m}" DataType="Byte" BlockXSize="256" BlockYSize="256"/>'
            f'<SrcRect xOff="0" yOff="0" xSize="{m}" ySize="{m}"/>'
            f'<DstRect xOff="{(c["x"] - vx0) * m}" yOff="{(c["y"] - vy0) * m}" xSize="{m}" ySize="{m}"/></SimpleSource>'
            for c in cell_index)
        return (f'<VRTDataset rasterXSize="{W}" rasterYSize="{H}"><SRS>{wkt}</SRS>'
                f'<GeoTransform>{t0.c!r}, {r!r}, 0, {t0.f!r}, 0, {-r!r}</GeoTransform>'
                f'<VRTRasterBand dataType="Byte" band="1"><NoDataValue>0</NoDataValue><ColorInterp>Gray</ColorInterp>'
                f'{overviews}{srcs}</VRTRasterBand></VRTDataset>')

    ov_tags = ""
    for L in range(1, n_ovr + 1):
        fn = f"iraq_crops_ovr{L}.vrt"
        open(os.path.join(A.out, fn), "w").write(vrt_xml(L))
        ov_tags += f'<Overview><SourceFilename relativeToVRT="1">{fn}</SourceFilename><SourceBand>1</SourceBand></Overview>'
    open(os.path.join(A.out, "iraq_crops.vrt"), "w").write(vrt_xml(0, ov_tags))

# ----------------------------------------------------------------------------- boundaries
GOV = [  # standard name, Arabic, Kurdish (Sorani), match keys
    ("Baghdad", "بغداد", "بەغدا", ["baghdad"]),
    ("Basra", "البصرة", "بەسرە", ["basra", "basrah"]),
    ("Nineveh", "نينوى", "نەینەوا", ["ninawa", "nineveh", "ninewa", "ninevah"]),
    ("Erbil", "أربيل", "هەولێر", ["erbil", "arbil", "irbil", "hawler"]),
    ("Duhok", "دهوك", "دهۆک", ["duhok", "dahuk", "dohuk", "dihok"]),
    ("Sulaymaniyah", "السليمانية", "سلێمانی", ["sulaymaniyah", "sulaimaniya", "sulaymaniya", "slemani", "sulaimaniyah"]),
    ("Halabja", "حلبجة", "هەڵەبجە", ["halabja", "halabjah"]),
    ("Kirkuk", "كركوك", "کەرکووک", ["kirkuk", "tamim", "tameem"]),
    ("Diyala", "ديالى", "دیالە", ["diyala", "diyalah"]),
    ("Salah ad-Din", "صلاح الدين", "سەلاحەدین", ["salahaddin", "saladin", "salahuddin", "saladdin", "salahaldin", "salahdin", "salahadin"]),
    ("Anbar", "الأنبار", "ئەنبار", ["anbar"]),
    ("Babil", "بابل", "بابل", ["babil", "babylon"]),
    ("Karbala", "كربلاء", "کەربەلا", ["karbala"]),
    ("Najaf", "النجف", "نەجەف", ["najaf"]),
    ("Qadisiyah", "القادسية", "قادسیە", ["qadisiyah", "qadisiyyah", "qadissiya", "diwaniya", "qadisiya"]),
    ("Wasit", "واسط", "واست", ["wasit"]),
    ("Maysan", "ميسان", "مەیسان", ["maysan", "missan", "maisan"]),
    ("Dhi Qar", "ذي قار", "زیقار", ["dhiqar", "thiqar", "zhiqar"]),
    ("Muthanna", "المثنى", "موسەننا", ["muthanna", "muthannia", "muthana"]),
]
KRI = {"Erbil", "Duhok", "Sulaymaniyah", "Halabja"}
# Districts whose governorate is set explicitly. geoBoundaries' governorate (ADM1) and district (ADM2) layers come from
# different sources and disagree around Baghdad, so a purely spatial match puts Baghdad's outer districts in Babil,
# Anbar and Diyala. Keys are normalised district names; every override applied is listed in stats.json.
DISTRICT_GOV = {
    "karkh": "Baghdad", "kadhmiyah": "Baghdad", "kadhimiya": "Baghdad", "kadhimiyah": "Baghdad", "kadhemiya": "Baghdad",
    "mahmoudiya": "Baghdad", "mahmudiya": "Baghdad", "mahmudiyah": "Baghdad", "madain": "Baghdad", "madaen": "Baghdad",
    "abughraib": "Baghdad", "tarmiya": "Baghdad", "tarmiyah": "Baghdad", "adhamiya": "Baghdad", "risafa": "Baghdad",
    "thawra": "Baghdad", "sadrcity": "Baghdad",
    "hindiya": "Karbala", "hindiyah": "Karbala", "ainaltamur": "Karbala", "aintamur": "Karbala",
    "halabja": "Halabja", "halabcha": "Halabja", "helebce": "Halabja",
}
# Districts on the line between the Kurdistan Region and the other governorates, pinned to the governorate
# geoBoundaries' outlines already give them (official governorate lines, not who administers them today),
# so a future boundary-file update can't move them silently. Readers can regroup them from the district rows.
DISTRICT_PINNED = {
    "makhmour": "Erbil", "makhmur": "Erbil",
    "aqra": "Nineveh", "akre": "Nineveh", "akra": "Nineveh",
    "shikhan": "Nineveh", "shekhan": "Nineveh", "sheikhan": "Nineveh",
    "kifri": "Diyala", "kifry": "Diyala",
    "khanaqin": "Diyala", "khanaqeen": "Diyala",
}


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\b(al|as|an|ad|ar|at|ath|governorate|province|muhafazat|muhafaza)\b", "", s)
    return re.sub(r"[^a-z]", "", s)


def gov_of(name):
    k = norm(name)
    for g in GOV:
        if any(m in k or k in m for m in g[3] if k):
            return g[0]
    return None


def load_geojson(path, level):
    if path:
        return json.load(open(path))
    import urllib.request
    url = None
    try:
        meta = json.load(urllib.request.urlopen(f"https://www.geoboundaries.org/api/current/gbOpen/IRQ/{level}/", timeout=60))
        url = meta["gjDownloadURL"]
    except Exception as e:
        print("  geoBoundaries API unavailable (", e, ") — using GitHub")
        url = f"https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/gbOpen/IRQ/{level}/geoBoundaries-IRQ-{level}.geojson"
    return json.load(urllib.request.urlopen(url, timeout=120))


from shapely.geometry import box, mapping, shape
from shapely.ops import transform as shp_transform, unary_union
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
import pyproj

adm1 = load_geojson(A.adm1, "ADM1")
adm2 = load_geojson(A.adm2, "ADM2")
nf = A.adm_name_field
govs = []
for f in adm1["features"]:
    g = gov_of(f["properties"].get(nf, ""))
    if g is None:
        print("  ⚠️ unmatched governorate name:", f["properties"].get(nf))
        g = str(f["properties"].get(nf))
    govs.append((g, shape(f["geometry"]).buffer(0)))
districts = []
overrides = []
pinned = []
for i, f in enumerate(adm2["features"]):
    geom = shape(f["geometry"]).buffer(0)
    dname = str(f["properties"].get(nf, f"district {i}"))
    spatial = max(govs, key=lambda gg: gg[1].intersection(geom).area)[0]     # governorate it overlaps most
    gov = DISTRICT_GOV.get(norm(dname)) or DISTRICT_PINNED.get(norm(dname)) or spatial
    if gov != spatial:
        overrides.append(f"{dname}: {spatial} → {gov}")
    if norm(dname) in DISTRICT_PINNED:
        pinned.append(f"{dname}: {gov}" + (f" (spatial match {spatial})" if gov != spatial else ""))
    districts.append({"id": len(districts) + 1, "name": dname, "gov": gov,
                      "region": "Kurdistan Region" if gov in KRI else "Federal Iraq (outside Kurdistan Region)",
                      "geom": geom})
print(f"Boundaries: {len(govs)} governorate polygons, {len(districts)} districts")
for o in overrides:
    print("  district governorate set explicitly:", o)
print("  border districts pinned:", "; ".join(pinned) or "none found")

# ----------------------------------------------------------------------------- statistics (native UTM, 10 m)
K = 256
counts = np.zeros((len(districts) + 1, K), np.int64)
px_area_m2 = None
cover_done = defaultdict(float); cover_plan = defaultdict(float)


def zone_poly(epsg):
    """UTM zone strip over Iraq's latitudes, densified so its meridians stay curved in UTM."""
    z = epsg - 32600
    w, e = -180 + (z - 1) * 6, -180 + z * 6
    lats = [24 + i * 0.05 for i in range(int((42 - 24) / 0.05) + 1)]
    ring = [(w, la) for la in lats] + [(e, la) for la in reversed(lats)] + [(w, lats[0])]
    from shapely.geometry import Polygon
    return Polygon(ring)


for t, f in plan_tiles.items():
    epsg = int(f["properties"]["epsg_utm"])
    # clipped to the tile's own UTM zone, like the pixel counts below: tiles from two zones overlap along
    # 42°E and 48°E, and summing both footprints double-counted processed/planned km² there
    tb = shape(json.loads(json.dumps(f["geometry"]))).intersection(zone_poly(epsg))
    for d in districts:
        a = tb.intersection(d["geom"])
        if not a.is_empty:
            pa = shp_transform(pyproj.Transformer.from_crs(4326, epsg, always_xy=True).transform, a).area / 1e6
            cover_plan[d["id"]] += pa
            if t in tiles:
                cover_done[d["id"]] += pa

for i, (t, (kind, f)) in enumerate(sorted(tiles.items())):
    codes, classes, crs, tr = read_tile(kind, f)
    classes_ref = classes_ref or classes
    h, w = codes.shape
    shapes = [(transform_geom("EPSG:4326", crs, mapping(d["geom"])), d["id"]) for d in districts
              if d["geom"].intersects(box(*transform_bounds(crs, "EPSG:4326", tr.c, tr.f + tr.e * h, tr.c + tr.a * w, tr.f)))]
    dist = rasterize(shapes, out_shape=(h, w), transform=tr, fill=0, dtype="int32") if shapes else np.zeros((h, w), np.int32)
    epsg = crs.to_epsg()
    inzone = rasterize([(transform_geom("EPSG:4326", crs, mapping(zone_poly(epsg))), 1)], out_shape=(h, w),
                       transform=tr, fill=0, dtype="uint8").astype(bool) if epsg and 32601 <= epsg <= 32660 else np.ones((h, w), bool)
    sel = inzone & (codes > 0)
    counts += np.bincount(dist[sel].astype(np.int64) * K + codes[sel], minlength=counts.size).reshape(counts.shape)
    px_area_m2 = abs(tr.a * tr.e)
    print(f"  stats {i + 1}/{len(tiles)} {t}")

ha_px = px_area_m2 / 1e4
crop_names = classes_ref


def summarise(rows):
    """rows: array (K,) of pixel counts -> dict of hectares."""
    out = {"cropland_ha": 0.0, "classes": {}}
    for ci, c in enumerate(crop_names):
        sure, less, doubt = (rows[ci + 1 + k * LESS] * ha_px for k in (0, 1, 2))
        tot = sure + less + doubt
        if tot:
            out["classes"][c] = {"ha": round(tot, 1), "ha_sure": round(sure, 1)}
            if c in CEREALS:
                out["classes"][c]["ha_cereal70"] = round(sure + less, 1)   # cereal probability ≥ 70 % and in a patch ≥ MIN_PATCH_HA
        out["cropland_ha"] += tot
    out["cropland_ha"] = round(out["cropland_ha"], 1)
    return out


gov_meta = {g[0]: {"ar": g[1], "ku": g[2]} for g in GOV}
D, G, RG = [], defaultdict(lambda: np.zeros(K, np.int64)), defaultdict(lambda: np.zeros(K, np.int64))
gcov = defaultdict(lambda: [0.0, 0.0]); rcov = defaultdict(lambda: [0.0, 0.0])
for d in districts:
    row = counts[d["id"]]
    G[d["gov"]] += row; RG[d["region"]] += row
    for agg, key in ((gcov, d["gov"]), (rcov, d["region"])):
        agg[key][0] += cover_done[d["id"]]; agg[key][1] += cover_plan[d["id"]]
    D.append({"name": d["name"], "gov": d["gov"], "region": d["region"], **summarise(row),
              "processed_km2": round(cover_done[d["id"]], 1), "planned_km2": round(cover_plan[d["id"]], 1)})


def pack(name, row, cov, extra=None):
    return {"name": name, **(extra or {}), **summarise(row), "processed_km2": round(cov[0], 1), "planned_km2": round(cov[1], 1)}


stats = {
    "season": A.season_label,
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "classes": crop_names, "confidence_threshold": CONF_THRESHOLD, "cereal_threshold": CEREAL_THRESHOLD, "min_patch_ha": MIN_PATCH_HA, "pixel_size_m": round(math.sqrt(px_area_m2), 2),
    "tiles_finished": len(tiles), "tiles_planned": len(plan_tiles),
    "tiles_by_run": dict(by_run),
    "outside_boundaries_ha": summarise(counts[0])["cropland_ha"],
    "boundaries": "Districts: geoBoundaries gbOpen IRQ ADM2 (CC BY 4.0). Governorate of each district: the one it overlaps most "
                  "in geoBoundaries ADM1, except these set explicitly (the two layers disagree around Baghdad; Halabja is a "
                  "governorate since 2025): " + ("; ".join(overrides) or "none"),
    "district_overrides": overrides,
    "district_pinned": pinned,
    "region_definition": "Kurdistan Region = Duhok, Erbil, Sulaymaniyah and Halabja governorates as drawn in the boundary file. "
                         "District rows let readers regroup areas administered differently.",
    "regions": [pack(r, RG[r], rcov[r]) for r in sorted(RG)],
    "governorates": sorted([pack(g, G[g], gcov[g], {"name_ar": gov_meta.get(g, {}).get("ar", g), "name_ku": gov_meta.get(g, {}).get("ku", g),
                                                    "region": "Kurdistan Region" if g in KRI else "Federal Iraq (outside Kurdistan Region)"})
                            for g in G], key=lambda r: -r["cropland_ha"]),
    "districts": sorted(D, key=lambda r: (r["gov"], r["name"])),
}
json.dump(stats, open(os.path.join(A.out, "stats.json"), "w"), ensure_ascii=False, indent=1)


def write_csv(fn, rows, keys):
    with open(os.path.join(A.out, fn), "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.writer(fh)
        w.writerow(keys + ["cropland_ha", "wheat_ha", "wheat_ha_cereal70", "wheat_ha_sure",
                           "barley_ha", "barley_ha_cereal70", "barley_ha_sure", "rye_oats_ha", "processed_km2", "planned_km2"])
        for r in rows:
            c = r["classes"]
            g = lambda cl, k: c.get(cl, {}).get(k, 0)
            w.writerow([r.get(k, "") for k in keys] + [r["cropland_ha"],
                       g("wheat", "ha"), g("wheat", "ha_cereal70"), g("wheat", "ha_sure"),
                       g("barley", "ha"), g("barley", "ha_cereal70"), g("barley", "ha_sure"),
                       round(g("rye", "ha") + g("oats", "ha"), 1), r["processed_km2"], r["planned_km2"]])


write_csv("stats_regions.csv", stats["regions"], ["name"])
write_csv("stats_governorates.csv", stats["governorates"], ["name", "name_ar", "region"])
write_csv("stats_districts.csv", stats["districts"], ["name", "gov", "region"])

# simplified governorate outlines for the page (AOI picker + map)
gfeat = []
gnames = sorted({d["gov"] for d in districts})
graw = [unary_union([d["geom"] for d in districts if d["gov"] == g]) for g in gnames]
# Simplified together, so neighbouring governorates keep one shared border: simplifying each on its own
# pulled shared borders apart along the rivers, and the 2021 comparison counted farmland in those gaps
# as "outside boundaries". Fallback (older Shapely): a much smaller tolerance, so gaps stay under ~50 m.
try:
    import shapely
    gsimp = list(shapely.coverage_simplify(graw, 0.003))
except Exception as e:
    print("  coverage_simplify unavailable (", e, ") — simplifying each governorate at 0.0005°")
    gsimp = [g.simplify(0.0005) for g in graw]
for g, geom in zip(gnames, gsimp):
    gj = json.loads(json.dumps(mapping(geom)), parse_float=lambda v: round(float(v), 4))
    gfeat.append({"type": "Feature", "properties": {"name": g, "name_ar": gov_meta.get(g, {}).get("ar", g),
                                                    "name_ku": gov_meta.get(g, {}).get("ku", g)}, "geometry": gj})
json.dump({"type": "FeatureCollection", "features": gfeat}, open(os.path.join(A.out, "boundaries.geojson"), "w"),
          ensure_ascii=False, separators=(",", ":"))

# ----------------------------------------------------------------------------- legend files (QGIS style + CSV)
def lighter(hexc, f=0.55):
    r, g, b = (int(hexc[i:i + 2], 16) for i in (1, 3, 5))
    return "#%02x%02x%02x" % tuple(int(v + (255 - v) * f) for v in (r, g, b))


entries = []
for i, c in enumerate(crop_names, 1):
    col = PALETTE.get(c, "#888888")
    label = c.replace("_", " ").capitalize()
    entries.append((i, col, label))
    entries.append((i + LESS, lighter(col), label + " (less sure)"))
    if c in CEREALS:
        entries.append((i + 2 * LESS, lighter(col, 0.8), label + " (doubtful cereal: probability < 70% or speck < 0.1 ha)"))
# Labels are XML-escaped: "probability < 70%" written raw broke the file in QGIS ("unexpected character").
qml_items = "".join(f'<paletteEntry value="{v}" color="{col}" alpha="255" label={quoteattr(lab)}/>' for v, col, lab in entries)
open(os.path.join(A.out, "iraq_crops.qml"), "w").write(
    '<!DOCTYPE qgis><qgis version="3.34" styleCategories="Symbology"><pipe><rasterrenderer type="paletted" band="1" opacity="1" nodataColor="">'
    f'<colorPalette>{qml_items}</colorPalette></rasterrenderer></pipe></qgis>')
with open(os.path.join(A.out, "legend.csv"), "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["code", "class", "flag", "color"])
    w.writerow([0, "not cropland / not processed", "", ""])
    for v, col, lab in entries:
        w.writerow([v, crop_names[(v % LESS) - 1], ["", "less sure", "doubtful cereal (<70% or speck)"][v // LESS], col])

# ----------------------------------------------------------------------------- index for the page
if not A.skip_cells:
    json.dump({
        "season": A.season_label, "generated_utc": stats["generated_utc"],
        "cell_zoom": CELL_Z, "cell_px": CELL_PX, "cell_res": CELL_RES, "max_zoom": MAX_Z,
        "overview": {"file": "overview.tif", "zoom": OVERVIEW_Z},
        "cells": cell_index, "vrt": "iraq_crops.vrt",
        "codes": {"not_cropland": 0, "class_offset": 1, "less_sure_offset": LESS, "classes": crop_names,
                  "colors": [PALETTE.get(c, "#888888") for c in crop_names],
                  "confidence_threshold": CONF_THRESHOLD, "doubtful_cereal_offset": 2 * LESS,
                  "cereal_threshold": CEREAL_THRESHOLD, "min_patch_ha": MIN_PATCH_HA, "cereals": list(CEREALS),
                  "cereal_probability": "P(wheat) + P(barley)"},
        "tiles_finished": len(tiles), "tiles_planned": len(plan_tiles), "tiles_by_run": dict(by_run),
        # outline of every processed tile (lon/lat), so viewers can tell "not cropland" from "not processed";
        # run and season say which run (and so which season window) the tile's pixels come from
        "processed": [{"tile": t, "run": sources[t]["run"], "season": sources[t]["season"],
                       "ring": [[round(x, 5), round(y, 5)] for x, y in plan_tiles[t]["geometry"]["coordinates"][0]]}
                      for t in sorted(tiles)],
        "license": "CC BY 4.0",
        "attribution": "Crop types: ESA WorldCereal system, run by Iraq Farms Health on Copernicus Data Space. "
                       "Contains modified Copernicus Sentinel data " + A.season_label.split("-")[0] + ".",
    }, open(os.path.join(A.out, "index.json"), "w"), indent=1)

tot = {r["name"]: r for r in stats["regions"]}
print("\nRegions:")
for r in stats["regions"]:
    c = r["classes"]
    print(f"  {r['name']}: wheat {c.get('wheat', {}).get('ha', 0):,.0f} ha, barley {c.get('barley', {}).get('ha', 0):,.0f} ha, "
          f"processed {r['processed_km2']:,.0f} of {r['planned_km2']:,.0f} km² planned")
size = sum(os.path.getsize(p) for p in glob.glob(os.path.join(A.out, "**", "*"), recursive=True) if os.path.isfile(p))
print(f"\nOutput: {size / 1e6:,.0f} MB in {A.out}")

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
!python /content/publish_crops.py "$RUN" "$OUT" --season-label "$SEASON_LABEL" {OVERLAY_ARGS}

## Save to Drive and check sizes
Copies the result to `MyDrive/iraq-farms-health/worldcereal/<RUN_NAME>/publish/<DATA_FOLDER>/` (replacing an older copy) and also makes one zip next to it.
GitHub's website accepts files up to **25 MB** and 100 files per upload; anything larger is flagged below.

In [ ]:
import glob, datetime
shutil.rmtree(DRIVE_OUT, ignore_errors=True)
shutil.copytree(OUT, DRIVE_OUT)
files = [f for f in glob.glob(OUT + "/**/*", recursive=True) if os.path.isfile(f)]
big = [(os.path.relpath(f, OUT), os.path.getsize(f) / 1e6) for f in files if os.path.getsize(f) > 25e6]
total = sum(os.path.getsize(f) for f in files) / 1e6
print(f"{len(files)} files, {total:,.0f} MB  →  {DRIVE_OUT}")
print(f"cells: {len(glob.glob(OUT + '/cells/*.tif'))}")
for name, mb in big:
    print(f"  ⚠️  {name}: {mb:.0f} MB — over GitHub's 25 MB web-upload limit (use GitHub Desktop for this one)")
if not big:
    print("✅ every file is under 25 MB — the GitHub website upload works")
zip_path = shutil.make_archive(f"{RUN}/publish/crops_{DATA_FOLDER}_{datetime.datetime.now(datetime.timezone.utc):%Y%m%d_%H%M}", "zip", "/content/publish", DATA_FOLDER)   # this output only, as <DATA_FOLDER>/...
print("Zip:", zip_path, f"({os.path.getsize(zip_path)/1e6:,.0f} MB)")